# Klasifikasi Uang Kertas Rupiah - MobileNetV2
Transfer learning MobileNetV2 dengan input gambar 224x224.

Alur pengerjaan:
1. Training awal (freeze backbone MobileNetV2, latih classifier head aja)
2. Fine-tuning (unfreeze beberapa layer atas biar lebih akurat)

## 1. Import Library

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# cek versi tensorflow dan device gpu
print('TensorFlow version:', tf.__version__)
print('GPU Device:', tf.config.list_physical_devices('GPU'))

## 2. Setup & Konfigurasi

In [ ]:
# path dataset di lokal
DATASET_BASE = r'C:\DL\rupiah-detector\dataset'

TRAIN_DIR = os.path.join(DATASET_BASE, 'train')
VAL_DIR   = os.path.join(DATASET_BASE, 'val')
TEST_DIR  = os.path.join(DATASET_BASE, 'test')

# setting gambar dan parameter training
IMG_SIZE         = (224, 224)
BATCH_SIZE       = 32
EPOCHS_FROZEN    = 15
EPOCHS_FINETUNE  = 20
LR_FROZEN        = 1e-3
LR_FINETUNE      = 1e-5
NUM_CLASSES      = 7
CLASSES          = ['1000', '2000', '5000', '10000', '20000', '50000', '100000']

# folder buat nyimpen hasil model
MODEL_DIR          = r'C:\DL\rupiah-detector\models'
MODEL_SAVE_PATH    = os.path.join(MODEL_DIR, 'mobilenetv2_frozen.keras')
FINETUNE_SAVE_PATH = os.path.join(MODEL_DIR, 'mobilenetv2_finetuned.keras')

os.makedirs(MODEL_DIR, exist_ok=True)
print('Dataset :', DATASET_BASE)
print('Models  :', MODEL_DIR)

## 3. Cek Jumlah Data per Kelas

In [ ]:
# fungsi buat ngitung isi gambar per folder kelas
def count_dataset(split_dir, split_name):
    if not os.path.exists(split_dir):
        print(f'Folder tidak ditemukan: {split_dir}')
        return {}
    total = 0
    counts = {}
    for cls in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls)
        if os.path.isdir(cls_path):
            n = len(list(pathlib.Path(cls_path).glob('*.[jJpP][pPnN][gGeE]*')))
            counts[cls] = n
            total += n
    print(f'{split_name} — total: {total}')
    for cls, n in counts.items():
        print(f'  {cls}: {n}')
    return counts

count_dataset(TRAIN_DIR, 'train')
count_dataset(VAL_DIR, 'val')
count_dataset(TEST_DIR, 'test')

## 4. Data Generator & Augmentasi

In [ ]:
# augmentasi khusus train biar variasinya nambah dan ga gampang overfit
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# val dan test cukup normalisasi pixel aja
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=True, seed=42
    )

val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', classes=CLASSES, shuffle=False
)

## 5. Cek Sample Gambar Hasil Augmentasi

In [ ]:
# ambil satu batch buat liat contoh gambarnya
images, labels = next(train_generator)
label_names = list(train_generator.class_indices.keys())

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
fig.suptitle('Sample gambar training (setelah augmentasi)')
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        ax.set_title(label_names[np.argmax(labels[i])])
    ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'sample_augmented.png'), dpi=100, bbox_inches='tight')
plt.show()

## 6. Buat Model (Transfer Learning MobileNetV2)

In [ ]:
def build_model_frozen(num_classes=7, img_size=(224, 224)):
    # load mobilenetv2 tanpa head classifier bawaan
    base_model = MobileNetV2(
        input_shape=(*img_size, 3),
        include_top=False,
        weights='imagenet'
    )
    # bekukan dulu layer-layer mobilenetv2
    base_model.trainable = False

    # tambah layer baru di atasnya khusus buat 7 pecahan rupiah
    inputs = tf.keras.Input(shape=(*img_size, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name='MobileNetV2_Rupiah')
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LR_FROZEN),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model, base_model

model, base_model = build_model_frozen(NUM_CLASSES, IMG_SIZE)
model.summary()

## 7. Training Tahap 1 (Backbone Dibekukan)

In [ ]:
# callbacks biar ga overfit dan otomatis simpen akurasi terbaik
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
]

history_frozen = model.fit(
    train_generator,
    epochs=EPOCHS_FROZEN,
    validation_data=val_generator,
    callbacks=callbacks,
    verbose=1
)

print('Model disimpan ke:', MODEL_SAVE_PATH)

## 8. Visualisasi Hasil Training

In [ ]:
def plot_history(history, title='Training History', save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title)

    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Val')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Val')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

plot_history(history_frozen, 'Tahap 1 — Backbone Frozen', os.path.join(MODEL_DIR, 'history_frozen.png'))

## 9. Fine-Tuning (Unfreeze Sebagian Layer)

Jalankan cell ini kalau Tahap 1 udah kelar.

In [ ]:
# buka 30 layer teratas mobilenetv2 buat dituning
UNFREEZE_LAYERS = 30
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

# compile ulang pake lr yang jauh lebih kecil biar bobotnya ga rusak
model.compile(
    optimizer=optimizers.Adam(learning_rate=LR_FINETUNE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ft = [
    EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    ModelCheckpoint(FINETUNE_SAVE_PATH, monitor='val_accuracy', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
]

history_ft = model.fit(
    train_generator,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_generator,
    callbacks=callbacks_ft,
    verbose=1
)

plot_history(history_ft, 'Tahap 2 — Fine-Tuning', os.path.join(MODEL_DIR, 'history_finetuned.png'))

## 10. Evaluasi Model di Test Set

In [ ]:
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

acc = accuracy_score(y_true, y_pred)
print(f'Test Accuracy: {acc:.4f} ({acc*100:.2f}%)')

target_names = [f'Rp {c}' for c in CLASSES]
print(classification_report(y_true, y_pred, target_names=target_names))

# plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=target_names, yticklabels=target_names, cmap='Blues')
plt.title('Confusion Matrix — MobileNetV2')
plt.ylabel('Aktual')
plt.xlabel('Prediksi')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'confusion_matrix_mobilenetv2.png'), dpi=120, bbox_inches='tight')
plt.show()

## 11. Simpan Model Akhir

In [ ]:
# copy atau simpan model yang mau dipake ke API
api_model_path = os.path.join(MODEL_DIR, 'mobilenetv2_rupiah.keras')
model.save(api_model_path)
print('Model final siap dipakai API:', api_model_path)